# Modelling & EDA
## Intelligent DSS — Early Risk Prediction of Type 2 Diabetes
---
**Workflow**
1. Imports
2. Load & inspect data
3. Clean & encode
4. EDA (distributions, correlation, class balance)
5. Train / validation split (80 / 20 from train.csv)
6. Scale features
7. SMOTE — fix class imbalance
8. Model training & cross-validation comparison
9. Final model evaluation (test set)
10. SHAP explainability
11. Save artefacts


## 1. Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pickle
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve,
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report,
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE

import shap
import mlflow

%matplotlib inline
print("All imports OK")


## 2. Constants & Column Lists

In [ ]:
RANDOM_STATE = 42
TARGET       = "diabetes_status"

# Columns we actually use (everything else is dropped)
KEEP_COLS = [
    "age", "sex", "is_pregnant",
    "bmi", "bmi_category",
    "family_history_diabetes", "previous_gdm",
    "physically_active", "has_hypertension",
    "fasting_glucose_mg_dl", "hba1c_percent",
    "total_cholesterol_mg_dl", "ldl_mg_dl",
    "hdl_mg_dl", "triglycerides_mg_dl",
    TARGET,
]

# Numeric columns that will be scaled
NUMERIC_COLS = [
    "age", "bmi",
    "fasting_glucose_mg_dl", "hba1c_percent",
    "total_cholesterol_mg_dl", "ldl_mg_dl",
    "hdl_mg_dl", "triglycerides_mg_dl",
]

# Categorical columns that need encoding
CATEGORICAL_COLS = [
    "sex", "is_pregnant", "bmi_category",
    "family_history_diabetes", "previous_gdm",
    "physically_active", "has_hypertension",
]

# Artefact paths
ARTEFACTS = "../../../artifacts"
import os
os.makedirs(ARTEFACTS, exist_ok=True)

print("Constants set.")


## 3. Load Data

We use **train.csv** (10 000 records) and **test.csv** (2 000 records).  
The provided validation.csv is ignored because its first 5 000 rows are  
identical to the first 5 000 rows of train.csv (data leakage).  
We create our own clean 80/20 validation split from train.csv in Section 5.


In [ ]:
df_train_raw = pd.read_csv("../../data/train.csv")
df_test_raw  = pd.read_csv("../../data/test.csv")

print(f"Train raw : {df_train_raw.shape}")
print(f"Test  raw : {df_test_raw.shape}")


In [ ]:
# Keep only the columns we need
df_train_raw = df_train_raw[KEEP_COLS].copy()
df_test_raw  = df_test_raw[KEEP_COLS].copy()

print(f"Train after column selection : {df_train_raw.shape}")
print(f"Test  after column selection : {df_test_raw.shape}")
df_train_raw.head()


## 4. Initial Inspection

In [ ]:
df_train_raw.dtypes


In [ ]:
df_train_raw.describe()


In [ ]:
print("Missing values — Train:")
print(df_train_raw.isnull().sum())
print()
print("Missing values — Test:")
print(df_test_raw.isnull().sum())


In [ ]:
print("Target value counts — Train:")
print(df_train_raw[TARGET].value_counts())


## 5. Clean & Encode

**Target encoding**
- `"Diabetic"` or `"Pre-diabetic"` → **1**  
- `"Normal"` → **0**

**Feature encoding**
- Boolean columns (True/False strings) → 1 / 0  
- `bmi_category` → ordinal integer (WHO classification)  
- `sex` → Female = 0, Male = 1


In [ ]:
# ── Target ───────────────────────────────────────────────────────────────────
def encode_target(df):
    df = df.copy()
    at_risk = {"diabetic", "pre-diabetic", "prediabetic"}
    df[TARGET] = (
        df[TARGET].astype(str).str.strip().str.lower()
        .apply(lambda v: 1 if any(r in v for r in at_risk) else 0)
    )
    return df

df_train_raw = encode_target(df_train_raw)
df_test_raw  = encode_target(df_test_raw)

print("Target after encoding:")
print(df_train_raw[TARGET].value_counts())


In [ ]:
# ── Boolean columns ───────────────────────────────────────────────────────────
BOOL_COLS = [
    "is_pregnant", "family_history_diabetes",
    "previous_gdm", "physically_active", "has_hypertension",
]

def encode_booleans(df):
    df = df.copy()
    for col in BOOL_COLS:
        if col in df.columns:
            df[col] = (
                df[col].astype(str).str.strip().str.lower()
                .map({"true": 1, "false": 0, "yes": 1, "no": 0, "1": 1, "0": 0})
                .fillna(0).astype(int)
            )
    return df

df_train_raw = encode_booleans(df_train_raw)
df_test_raw  = encode_booleans(df_test_raw)

print("Boolean columns encoded.")
df_train_raw[BOOL_COLS].head(3)


In [ ]:
# ── bmi_category — ordinal ────────────────────────────────────────────────────
BMI_MAP = {
    "underweight": 0,
    "normal"     : 1,
    "overweight" : 2,
    "obese i"    : 3,
    "obese ii"   : 4,
    "obese iii"  : 5,
}

def encode_bmi_category(df):
    df = df.copy()
    df["bmi_category"] = (
        df["bmi_category"].astype(str).str.strip().str.lower()
        .map(BMI_MAP).fillna(1).astype(int)
    )
    return df

df_train_raw = encode_bmi_category(df_train_raw)
df_test_raw  = encode_bmi_category(df_test_raw)

print("bmi_category unique values:", sorted(df_train_raw["bmi_category"].unique()))


In [ ]:
# ── sex ───────────────────────────────────────────────────────────────────────
SEX_MAP = {"female": 0, "male": 1}

def encode_sex(df):
    df = df.copy()
    df["sex"] = (
        df["sex"].astype(str).str.strip().str.lower()
        .map(SEX_MAP).fillna(0).astype(int)
    )
    return df

df_train_raw = encode_sex(df_train_raw)
df_test_raw  = encode_sex(df_test_raw)

print("sex unique values:", df_train_raw["sex"].unique())


In [ ]:
# ── Impute remaining nulls with median / mode ────────────────────────────────
# Fit statistics on full training data, apply to both splits

median_fills = {col: df_train_raw[col].median() for col in NUMERIC_COLS
                if col in df_train_raw.columns}
mode_fills   = {col: df_train_raw[col].mode()[0] for col in CATEGORICAL_COLS
                if col in df_train_raw.columns}

def impute(df, median_fills, mode_fills):
    df = df.copy()
    for col, val in median_fills.items():
        df[col] = df[col].fillna(val)
    for col, val in mode_fills.items():
        df[col] = df[col].fillna(val)
    return df

df_train_clean = impute(df_train_raw, median_fills, mode_fills)
df_test_clean  = impute(df_test_raw,  median_fills, mode_fills)

remaining = df_train_clean.isnull().sum().sum()
print(f"Nulls remaining after imputation: {remaining}")


## 6. Exploratory Data Analysis

In [ ]:
# Class distribution
counts = df_train_clean[TARGET].value_counts()
pcts   = (counts / len(df_train_clean) * 100).round(1)

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(["Normal (0)", "Diabetic (1)"], counts.values,
              color=["#2E75B6", "#C0392B"], edgecolor="white", width=0.5)
for bar, pct, n in zip(bars, pcts.values, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f"{n:,}\n({pct}%)", ha="center", fontsize=10, fontweight="bold")
ax.set_title("Target Class Distribution — Training Data", fontweight="bold")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()


In [ ]:
# Distribution of numeric features split by target
key_cols = ["age", "bmi", "fasting_glucose_mg_dl",
            "hba1c_percent", "triglycerides_mg_dl", "hdl_mg_dl"]
key_cols = [c for c in key_cols if c in df_train_clean.columns]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(key_cols):
    ax = axes[i]
    for cls, color, label in [(0, "#2E75B6", "Normal"), (1, "#C0392B", "Diabetic")]:
        data = df_train_clean.loc[df_train_clean[TARGET] == cls, col].dropna()
        ax.hist(data, bins=35, alpha=0.55, color=color,
                label=label, edgecolor="none", density=True)
    ax.set_title(col, fontweight="bold")
    ax.legend(fontsize=8)

plt.suptitle("Feature Distributions by Diabetes Status", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# Correlation heatmap
numeric_in = [c for c in NUMERIC_COLS if c in df_train_clean.columns] + [TARGET]
corr = df_train_clean[numeric_in].corr()

plt.figure(figsize=(11, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.4, annot_kws={"size": 9})
plt.title("Pearson Correlation — Numeric Features", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# Boxplots: key numeric features vs target
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, col in enumerate(key_cols):
    ax = axes[i]
    d0 = df_train_clean.loc[df_train_clean[TARGET] == 0, col].dropna()
    d1 = df_train_clean.loc[df_train_clean[TARGET] == 1, col].dropna()
    bp = ax.boxplot([d0, d1], patch_artist=True,
                    labels=["Normal", "Diabetic"], widths=0.45,
                    medianprops=dict(color="#C0392B", linewidth=2.5),
                    flierprops=dict(marker="o", markersize=2, alpha=0.25))
    bp["boxes"][0].set_facecolor("#D6E4F0")
    bp["boxes"][1].set_facecolor("#FADBD8")
    ax.set_title(col, fontweight="bold")

plt.suptitle("Numeric Features by Diabetes Status", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# Pairplot (sampled for speed)
sample = df_train_clean[key_cols + [TARGET]].sample(800, random_state=RANDOM_STATE)
sample[TARGET] = sample[TARGET].map({0: "Normal", 1: "Diabetic"})
sns.pairplot(sample, hue=TARGET, palette={"Normal": "#2E75B6", "Diabetic": "#C0392B"},
             plot_kws={"alpha": 0.35, "edgecolor": "none"}, diag_kind="kde")
plt.suptitle("Pairplot — Key Numeric Features (800-sample)", y=1.01, fontweight="bold")
plt.show()


## 7. Train / Validation Split

Split `train.csv` into **80% train** and **20% validation** using stratified
sampling so the class ratio is preserved in both halves.


In [ ]:
FEATURE_COLS = [c for c in KEEP_COLS if c != TARGET]

X = df_train_clean[FEATURE_COLS].values.astype(float)
y = df_train_clean[TARGET].values.astype(int)

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_test = df_test_clean[FEATURE_COLS].values.astype(float)
y_test = df_test_clean[TARGET].values.astype(int)

print(f"X_train : {X_train.shape}   class dist: {np.bincount(y_train)}")
print(f"X_val   : {X_val.shape}   class dist: {np.bincount(y_val)}")
print(f"X_test  : {X_test.shape}   class dist: {np.bincount(y_test)}")


## 8. Feature Scaling

`MinMaxScaler` maps every numeric feature to [0, 1].  
Fitted on **X_train only** — then applied to val and test.


In [ ]:
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

print("Scaling applied. Sample ranges after scaling (X_train):")
for i, col in enumerate(FEATURE_COLS):
    lo, hi = X_train[:, i].min(), X_train[:, i].max()
    print(f"  {col:<32}: {lo:.4f} – {hi:.4f}")


## 9. SMOTE — Fix Class Imbalance

Applied to **training set only**. Creates synthetic minority-class samples
so both classes are equally represented before model fitting.


In [ ]:
smote = SMOTE(k_neighbors=5, random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f"Before SMOTE : {np.bincount(y_train)}   total = {len(y_train):,}")
print(f"After  SMOTE : {np.bincount(y_train_res)}   total = {len(y_train_res):,}")


## 10. Model Comparison — 5-Fold Cross-Validation

Five candidate classifiers are trained and compared using stratified 5-fold CV
on the resampled training set.

**Primary metric:** AUC-ROC  
**Tiebreaker:** Recall — missing a diabetic patient is the worst error


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, C=1.0, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=10, min_samples_leaf=5,
        class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric="logloss", use_label_encoder=False,
        random_state=RANDOM_STATE, verbosity=0
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=6,
        num_leaves=63, class_weight="balanced",
        random_state=RANDOM_STATE, verbose=-1
    ),
    "CatBoost": CatBoostClassifier(
        iterations=200, learning_rate=0.05, depth=6,
        class_weights={0: 1, 1: 3}, random_seed=RANDOM_STATE, verbose=0
    ),
}

scoring = {
    "roc_auc"  : "roc_auc",
    "recall"   : "recall",
    "precision": "precision",
    "f1"       : "f1",
    "accuracy" : "accuracy",
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = {}

for name, model in models.items():
    print(f"  Training {name} ...", end=" ", flush=True)
    out = cross_validate(model, X_train_res, y_train_res,
                         cv=skf, scoring=scoring, n_jobs=-1)
    cv_results[name] = {m: {"mean": out[f"test_{m}"].mean(),
                             "std" : out[f"test_{m}"].std()}
                        for m in scoring}
    auc = cv_results[name]["roc_auc"]["mean"]
    rec = cv_results[name]["recall"]["mean"]
    print(f"AUC={auc:.4f}  Recall={rec:.4f}  ✅")

print("\nDone.")


In [ ]:
# Comparison table
rows = []
for name, scores in cv_results.items():
    row = {"Model": name}
    for m in ["roc_auc", "recall", "precision", "f1", "accuracy"]:
        row[f"{m}_mean"] = round(scores[m]["mean"], 4)
        row[f"{m}_std"]  = round(scores[m]["std"],  4)
    rows.append(row)

cv_df = (pd.DataFrame(rows)
         .sort_values(["roc_auc_mean", "recall_mean"], ascending=False)
         .reset_index(drop=True))

mean_cols = [c for c in cv_df.columns if c.endswith("_mean")]
cv_df.style \
    .highlight_max(subset=mean_cols, color="#C6EFCE") \
    .format({c: "{:.4f}" for c in cv_df.select_dtypes(float).columns})


In [ ]:
# Bar chart comparison
metrics_plot = ["roc_auc_mean", "recall_mean", "f1_mean", "precision_mean"]
model_names  = cv_df["Model"].tolist()
x      = np.arange(len(model_names))
width  = 0.18
colors = ["#1F3864", "#2E75B6", "#70AD47", "#ED7D31"]

fig, ax = plt.subplots(figsize=(13, 5))
for i, (m, c) in enumerate(zip(metrics_plot, colors)):
    offset = (i - len(metrics_plot)/2 + 0.5) * width
    ax.bar(x + offset, cv_df[m].values, width, label=m.replace("_mean",""),
           color=c, edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=15, ha="right")
ax.set_ylim(0.4, 1.05)
ax.set_ylabel("Score")
ax.set_title("5-Fold CV — Candidate Model Comparison", fontweight="bold")
ax.legend(loc="lower right")
ax.axhline(0.85, color="#C0392B", linestyle="--", linewidth=0.9, label="AUC target")
plt.tight_layout()
plt.show()


## 11. Aggregated Confusion Matrix — Best Model (5-Fold CV)

Retrain the best model fold-by-fold and accumulate confusion matrices,
matching the style from the reference notebook.


In [ ]:
# Identify best model
best_name  = cv_df.iloc[0]["Model"]
best_model = models[best_name]
print(f"Best model: {best_name}")

# 5-fold aggregated confusion matrix
final_cm   = np.zeros((2, 2), dtype=int)
accuracies = []

skf2 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for train_idx, val_idx in skf2.split(X_train_res, y_train_res):
    Xf_tr, Xf_val = X_train_res[train_idx], X_train_res[val_idx]
    yf_tr, yf_val = y_train_res[train_idx], y_train_res[val_idx]

    import copy
    fold_model = copy.deepcopy(best_model)
    fold_model.fit(Xf_tr, yf_tr)
    yf_hat = fold_model.predict(Xf_val)

    accuracies.append(accuracy_score(yf_val, yf_hat))
    final_cm += confusion_matrix(yf_val, yf_hat, labels=[0, 1])

print(f"Mean Accuracy : {np.mean(accuracies):.4f}")
print(f"Std  Accuracy : {np.std(accuracies):.4f}")

plt.figure(figsize=(6, 5))
sns.heatmap(final_cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Normal", "Diabetic"],
            yticklabels=["Normal", "Diabetic"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Aggregated Confusion Matrix — {best_name} (5-Fold CV)")
plt.tight_layout()
plt.show()


## 12. Final Model — Train on Full Set, Evaluate on Validation

Retrain on the **entire resampled training set**, then evaluate on the
clean validation split (which was never seen during cross-validation).


In [ ]:
import copy
final_model = copy.deepcopy(best_model)
final_model.fit(X_train_res, y_train_res)

y_val_prob = final_model.predict_proba(X_val)[:, 1]
y_val_pred = (y_val_prob >= 0.5).astype(int)

print("Validation Set Performance (threshold = 0.5):")
print(f"  AUC-ROC   : {roc_auc_score(y_val, y_val_prob):.4f}")
print(f"  Recall    : {recall_score(y_val, y_val_pred):.4f}")
print(f"  Precision : {precision_score(y_val, y_val_pred, zero_division=0):.4f}")
print(f"  F1        : {f1_score(y_val, y_val_pred):.4f}")
print(f"  Accuracy  : {accuracy_score(y_val, y_val_pred):.4f}")


## 13. Threshold Tuning — Youden's J Index

The default 0.5 threshold is not optimal for imbalanced medical data.  
Youden's J = Sensitivity + Specificity − 1.  
We pick the threshold that maximises J on the validation set.


In [ ]:
fpr, tpr, thresholds = roc_curve(y_val, y_val_prob)
youden_j  = tpr - fpr
best_idx  = int(np.argmax(youden_j))
OPT_THRESH = float(thresholds[best_idx])

THRESH_LOW_HIGH  = OPT_THRESH
THRESH_HIGH      = min(OPT_THRESH + 0.20, 0.90)

print(f"Optimal threshold (Youden J = {youden_j[best_idx]:.4f}): {OPT_THRESH:.4f}")
print(f"  Sensitivity : {tpr[best_idx]:.4f}")
print(f"  Specificity : {1 - fpr[best_idx]:.4f}")
print()
print("Risk bands:")
print(f"  Low      : P < {THRESH_LOW_HIGH:.4f}")
print(f"  Moderate : {THRESH_LOW_HIGH:.4f} <= P < {THRESH_HIGH:.4f}")
print(f"  High     : P >= {THRESH_HIGH:.4f}")

# Plot
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr, tpr, color="#1F3864", linewidth=2,
        label=f"AUC = {roc_auc_score(y_val, y_val_prob):.4f}")
ax.scatter(fpr[best_idx], tpr[best_idx], color="#C0392B", s=100, zorder=5,
           label=f"Optimal threshold = {OPT_THRESH:.3f}")
ax.plot([0,1],[0,1],"k--",linewidth=0.8)
ax.set_xlabel("1 - Specificity")
ax.set_ylabel("Sensitivity")
ax.set_title("ROC Curve — Validation Set", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()


## 14. Final Evaluation — Test Set

The test set (2 000 records) has never been seen at any point.  
This gives the unbiased estimate of real-world performance.


In [ ]:
y_test_prob = final_model.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_prob >= OPT_THRESH).astype(int)

test_auc  = roc_auc_score(y_test, y_test_prob)
test_rec  = recall_score(y_test, y_test_pred)
test_prec = precision_score(y_test, y_test_pred, zero_division=0)
test_f1   = f1_score(y_test, y_test_pred)
test_acc  = accuracy_score(y_test, y_test_pred)

print("Test Set Performance:")
print(f"  AUC-ROC   : {test_auc:.4f}")
print(f"  Recall    : {test_rec:.4f}")
print(f"  Precision : {test_prec:.4f}")
print(f"  F1        : {test_f1:.4f}")
print(f"  Accuracy  : {test_acc:.4f}")
print()
print(classification_report(y_test, y_test_pred,
                             target_names=["Normal (0)", "Diabetic (1)"]))


In [ ]:
# Confusion matrices — counts and normalised
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

cm = confusion_matrix(y_test, y_test_pred)
ConfusionMatrixDisplay(cm, display_labels=["Normal", "Diabetic"]) \
    .plot(ax=ax1, colorbar=False, cmap="Blues")
ax1.set_title("Confusion Matrix — Counts", fontweight="bold")

cm_norm = confusion_matrix(y_test, y_test_pred, normalize="true")
ConfusionMatrixDisplay(cm_norm, display_labels=["Normal", "Diabetic"]) \
    .plot(ax=ax2, colorbar=False, cmap="Blues", values_format=".2%")
ax2.set_title("Confusion Matrix — Normalised", fontweight="bold")

plt.suptitle(f"Final Model: {best_name} — Test Set", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# ROC curve — test set
fpr_t, tpr_t, _ = roc_curve(y_test, y_test_prob)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr_t, tpr_t, color="#1F3864", linewidth=2,
        label=f"{best_name}  AUC = {test_auc:.4f}")
ax.plot([0,1],[0,1],"k--",linewidth=0.8)
ax.set_xlabel("1 - Specificity")
ax.set_ylabel("Sensitivity")
ax.set_title("ROC Curve — Test Set", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()


## 15. SHAP — Feature Importance & Explainability

**Global** — which features drive the model most on average  
**Local** — why did this specific patient get their risk score


In [ ]:
# Build explainer
tree_models = ("XGBoost", "LightGBM", "CatBoost", "Random Forest")

if best_name in tree_models:
    explainer = shap.TreeExplainer(final_model)
else:
    explainer = shap.LinearExplainer(final_model, X_train_res,
                                     feature_perturbation="interventional")

# Compute SHAP values on test set (cap at 1500 for speed)
N_SHAP = min(1500, len(X_test))
shap_vals = explainer.shap_values(X_test[:N_SHAP])

# Some explainers return [neg_class, pos_class]
if isinstance(shap_vals, list):
    shap_vals = shap_vals[1]

print(f"SHAP values shape: {shap_vals.shape}")


In [ ]:
# Global bar chart — mean |SHAP|
plt.figure(figsize=(9, 6))
shap.summary_plot(shap_vals, X_test[:N_SHAP],
                  feature_names=FEATURE_COLS,
                  plot_type="bar", show=False,
                  color="#2E75B6", max_display=len(FEATURE_COLS))
plt.title("Global Feature Importance — Mean |SHAP Value|", fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# Beeswarm — direction and magnitude
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_vals, X_test[:N_SHAP],
                  feature_names=FEATURE_COLS,
                  show=False, max_display=len(FEATURE_COLS))
plt.title("SHAP Beeswarm — Direction & Magnitude", fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# Local waterfall — single patient
PATIENT_IDX = 0
prob_p = float(final_model.predict_proba(X_test[PATIENT_IDX:PATIENT_IDX+1])[0, 1])

if prob_p >= THRESH_HIGH:
    risk_label = "HIGH RISK"
elif prob_p >= THRESH_LOW_HIGH:
    risk_label = "MODERATE RISK"
else:
    risk_label = "LOW RISK"

print(f"Patient {PATIENT_IDX}  |  Probability: {prob_p:.4f}  |  {risk_label}")

base = explainer.expected_value
if isinstance(base, list):
    base = base[1]

plt.figure(figsize=(10, 5))
shap.waterfall_plot(
    shap.Explanation(
        values       = shap_vals[PATIENT_IDX],
        base_values  = float(base),
        data         = X_test[PATIENT_IDX],
        feature_names= FEATURE_COLS,
    ),
    show=False, max_display=15,
)
plt.title(f"SHAP Waterfall — Patient {PATIENT_IDX}  ({risk_label})",
          fontweight="bold")
plt.tight_layout()
plt.show()


## 16. Save Artefacts

In [ ]:
def save(obj, path, label):
    with open(path, "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
    size = os.path.getsize(path) / 1024
    print(f"  ✅  {label:<28} → {os.path.basename(path)}  ({size:.1f} KB)")

save(final_model, f"{ARTEFACTS}/model.pkl",          "Final model")
save(scaler,      f"{ARTEFACTS}/scaler.pkl",         "MinMaxScaler")
save(explainer,   f"{ARTEFACTS}/shap_explainer.pkl", "SHAP Explainer")

# Feature + threshold config
cfg = {
    "model_name"       : best_name,
    "feature_cols"     : FEATURE_COLS,
    "numeric_cols"     : NUMERIC_COLS,
    "categorical_cols" : CATEGORICAL_COLS,
    "bool_cols"        : BOOL_COLS,
    "bmi_category_map" : BMI_MAP,
    "sex_map"          : SEX_MAP,
    "target_col"       : TARGET,
    "threshold_moderate": round(THRESH_LOW_HIGH, 6),
    "threshold_high"   : round(THRESH_HIGH, 6),
    "optimal_threshold": round(OPT_THRESH, 6),
    "median_fills"     : {k: float(v) for k, v in median_fills.items()},
    "test_metrics"     : {
        "auc_roc"  : round(test_auc,  4),
        "recall"   : round(test_rec,  4),
        "precision": round(test_prec, 4),
        "f1"       : round(test_f1,   4),
        "accuracy" : round(test_acc,  4),
    },
}

cfg_path = f"{ARTEFACTS}/feature_config.json"
with open(cfg_path, "w") as f:
    json.dump(cfg, f, indent=2)
print(f"  ✅  {'Config JSON':<28} → feature_config.json")

print()
print("All artefacts saved:")
for p in sorted(os.listdir(ARTEFACTS)):
    sz = os.path.getsize(f"{ARTEFACTS}/{p}") / 1024
    print(f"    {p:<35} ({sz:.1f} KB)")


In [ ]:
print("=" * 50)
print(f"  FINAL SUMMARY")
print("=" * 50)
print(f"  Model     : {best_name}")
print(f"  Features  : {len(FEATURE_COLS)}")
print(f"  Threshold : {OPT_THRESH:.4f}  (Youden J)")
print(f"  AUC-ROC   : {test_auc:.4f}")
print(f"  Recall    : {test_rec:.4f}")
print(f"  F1        : {test_f1:.4f}")
print("=" * 50)
print("  ✅  Ready for Kedro pipeline + FastAPI backend.")
